In [1]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.neighbors import KNeighborsClassifier

In [2]:
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
CLASS_NAMES = ["Low", "Medium", "High"]

DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

In [3]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

preprocessor = joblib.load(MODELS_DIR / "preprocessor.joblib")
X_train_prepared = preprocessor.transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f"Training set: {X_train_prepared.shape}")
print(f"Test set: {X_test_prepared.shape}")

Training set: (1600, 17)
Test set: (400, 17)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_prepared, y_train)
    y_pred = model.predict(X_test_prepared)
    y_probability = model.predict_proba(X_test_prepared)

    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "ROC-AUC (OvR macro)": roc_auc_score(
            y_test, y_probability, multi_class="ovr", average="macro"
        ),
    })

    print(f"{name} trained.")

Logistic Regression trained.
K-Nearest Neighbors trained.
Random Forest trained.


In [7]:
comparison = pd.DataFrame(results).sort_values("F1-score (macro)", ascending=False)
display(comparison.round(3))

,Model,Accuracy,Precision (macro),Recall (macro),F1-score (macro),ROC-AUC (OvR macro)
0,Logistic Regression,0.778,0.779,0.777,0.778,0.917
2,Random Forest,0.678,0.672,0.676,0.674,0.868
1,K-Nearest Neighbors,0.588,0.603,0.587,0.591,0.770
